## Imports e Configuração Inicial

In [1]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona pasta src ao path
sys.path.append(os.path.abspath(os.path.join('..')))

from src.dataset import get_svhn_loaders
from src.model import SVHNNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Carrega os dados (batch_size maior ajuda a estabilizar gradientes para visualizar schedulers)
train_loader, test_loader = get_svhn_loaders(batch_size=128)

Device: cpu
Using downloaded and verified file: ./data/train_32x32.mat
Using downloaded and verified file: ./data/test_32x32.mat


## Função de Treino com Registro de LR

Esta função é "híbrida": ela aceita um argumento step_on_batch para saber se deve atualizar a LR a cada iteração ou só no final da época.


In [2]:
def train_with_scheduler(model, optimizer, scheduler, scheduler_name, epochs=10, step_on_batch=False):
    criterion = nn.CrossEntropyLoss()
    
    # Históricos
    lr_history = []
    loss_history = []
    val_acc_history = []
    
    print(f"--- Iniciando Treino com {scheduler_name} ---")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Captura a LR atual
            current_lr = optimizer.param_groups[0]['lr']
            lr_history.append(current_lr)
            
            # Se o scheduler for por batch (ex: OneCycleLR, CyclicLR), atualiza aqui
            if step_on_batch:
                scheduler.step()
        
        # Se o scheduler for por época (ex: StepLR), atualiza aqui
        if not step_on_batch:
            scheduler.step()
            
        # Validação rápida
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_acc = 100 * correct / total
        val_acc_history.append(val_acc)
        avg_loss = running_loss / len(train_loader)
        loss_history.append(avg_loss)
        
        print(f"Epoch [{epoch+1}/{epochs}] LR: {current_lr:.6f} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}%")
        
    return lr_history, loss_history, val_acc_history

## Experimento 1 - StepLR (O Clássico)
Aqui vamos usar o StepLR. Ele mantém a LR constante e a derruba drasticamente a cada X épocas. É ótimo para "refinar" o modelo quando ele para de aprender.

In [3]:
# Setup Experimento 1: StepLR
model_step = SVHNNet().to(device)
# Usamos SGD com Momentum alto para ver bem o efeito
optimizer_step = optim.SGD(model_step.parameters(), lr=0.1, momentum=0.9)

# Decai a LR por um fator de 0.1 a cada 5 épocas
scheduler_step = optim.lr_scheduler.StepLR(optimizer_step, step_size=5, gamma=0.1)

# Executa (step_on_batch=False pois StepLR é por época)
lr_hist_step, loss_hist_step, acc_hist_step = train_with_scheduler(
    model_step, 
    optimizer_step, 
    scheduler_step, 
    "StepLR", 
    epochs=15, 
    step_on_batch=False
)

--- Iniciando Treino com StepLR ---
Epoch [1/15] LR: 0.100000 | Loss: 1.2174 | Val Acc: 76.72%
Epoch [2/15] LR: 0.100000 | Loss: 0.4668 | Val Acc: 85.76%
Epoch [3/15] LR: 0.100000 | Loss: 0.3637 | Val Acc: 84.65%
Epoch [4/15] LR: 0.100000 | Loss: 0.3166 | Val Acc: 88.64%
Epoch [5/15] LR: 0.100000 | Loss: 0.2893 | Val Acc: 88.51%
Epoch [6/15] LR: 0.010000 | Loss: 0.2061 | Val Acc: 91.16%
Epoch [7/15] LR: 0.010000 | Loss: 0.1868 | Val Acc: 91.22%
Epoch [8/15] LR: 0.010000 | Loss: 0.1780 | Val Acc: 91.29%
Epoch [9/15] LR: 0.010000 | Loss: 0.1714 | Val Acc: 91.35%
Epoch [10/15] LR: 0.010000 | Loss: 0.1654 | Val Acc: 91.27%
Epoch [11/15] LR: 0.001000 | Loss: 0.1537 | Val Acc: 91.33%
Epoch [12/15] LR: 0.001000 | Loss: 0.1530 | Val Acc: 91.31%
Epoch [13/15] LR: 0.001000 | Loss: 0.1519 | Val Acc: 91.38%
Epoch [14/15] LR: 0.001000 | Loss: 0.1517 | Val Acc: 91.41%
Epoch [15/15] LR: 0.001000 | Loss: 0.1510 | Val Acc: 91.39%


## Experimento 2 - MultiStepLR (O Moderno)
Aqui usamos o MultiStepLR. Ele realiza o cálculo de atualização da Learning Rate (LR) apenas uma vez por época. Ele funciona exatamente como o StepLR que vimos antes: não precisa monitorar métricas de validação (como o ReduceLROnPlateau precisaria) nem definir funções complexas (como o LambdaLR).

In [4]:
# Setup Experimento 2: MultiStepLR
# Reiniciamos o modelo e otimizador para uma comparação justa
model_multistep = SVHNNet().to(device)
optimizer_multistep = optim.SGD(model_multistep.parameters(), lr=0.01, momentum=0.9)

# Configuração do MultiStepLR
# milestones: Lista de épocas onde a LR vai cair. 
# Ex: Caindo nas épocas 8 e 12 (de um total de 15)
# gamma: Fator de multiplicação (0.1 = divide por 10)
scheduler_multistep = optim.lr_scheduler.MultiStepLR(
    optimizer_multistep, 
    milestones=[8, 12], 
    gamma=0.1
)

# Executa
# step_on_batch=False -> Pois o MultiStepLR atualiza por época, não por batch
lr_hist_multistep, loss_hist_multistep, acc_hist_multistep = train_with_scheduler(
    model_multistep, 
    optimizer_multistep, 
    scheduler_multistep, 
    "MultiStepLR", 
    epochs=15, 
    step_on_batch=False
)

--- Iniciando Treino com MultiStepLR ---
Epoch [1/15] LR: 0.010000 | Loss: 0.7688 | Val Acc: 85.53%
Epoch [2/15] LR: 0.010000 | Loss: 0.3792 | Val Acc: 88.08%
Epoch [3/15] LR: 0.010000 | Loss: 0.3160 | Val Acc: 89.87%
Epoch [4/15] LR: 0.010000 | Loss: 0.2744 | Val Acc: 89.80%
Epoch [5/15] LR: 0.010000 | Loss: 0.2467 | Val Acc: 90.73%
Epoch [6/15] LR: 0.010000 | Loss: 0.2205 | Val Acc: 91.49%
Epoch [7/15] LR: 0.010000 | Loss: 0.2019 | Val Acc: 90.86%
Epoch [8/15] LR: 0.010000 | Loss: 0.1857 | Val Acc: 91.38%
Epoch [9/15] LR: 0.001000 | Loss: 0.1313 | Val Acc: 92.39%
Epoch [10/15] LR: 0.001000 | Loss: 0.1207 | Val Acc: 92.37%
Epoch [11/15] LR: 0.001000 | Loss: 0.1163 | Val Acc: 92.45%
Epoch [12/15] LR: 0.001000 | Loss: 0.1128 | Val Acc: 92.37%
Epoch [13/15] LR: 0.000100 | Loss: 0.1065 | Val Acc: 92.47%
Epoch [14/15] LR: 0.000100 | Loss: 0.1063 | Val Acc: 92.43%
Epoch [15/15] LR: 0.000100 | Loss: 0.1059 | Val Acc: 92.48%


## Visualização Comparativa

In [ ]:
# Configuração dos plots
fig, ax = plt.subplots(1, 3, figsize=(20, 5))

# 1. Evolução da Learning Rate
ax[0].plot(lr_hist_step, label='StepLR', color='blue')
# Precisamos ajustar a escala x do StepLR para bater com o OneCycle (que tem mais pontos) se quisermos sobrepor perfeitamente,
# mas plotar cru já mostra a diferença de comportamento (escada vs onda).
ax[0].plot(lr_hist_cycle, label='OneCycleLR', color='orange', alpha=0.7)
ax[0].set_title("Evolução da Learning Rate")
ax[0].set_xlabel("Steps (Batches)")
ax[0].set_ylabel("LR")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

# 2. Curvas de Loss
ax[1].plot(loss_hist_step, label='StepLR', marker='o')
ax[1].plot(loss_hist_cycle, label='OneCycleLR', marker='o')
ax[1].set_title("Training Loss por Época")
ax[1].set_xlabel("Época")
ax[1].set_ylabel("Loss")
ax[1].legend()
ax[1].grid(True)

# 3. Acurácia de Validação
ax[2].plot(acc_hist_step, label='StepLR', marker='o')
ax[2].plot(acc_hist_cycle, label='OneCycleLR', marker='o')
ax[2].set_title("Acurácia de Validação (%)")
ax[2].set_xlabel("Época")
ax[2].set_ylabel("Acurácia")
ax[2].legend()
ax[2].grid(True)

plt.tight_layout()
plt.show()